# WO-04c — fill the dummy from real data
Fills every `yes` unit of `modal_dummy_annotated.md` for the ten test places (no LLM), leaving each `maybe` and `no` unit as a visible placeholder naming what is missing; each result is a *filled_profile*, written to `docs/design/dashboard/wo04_filled_profiles.md`. Then counts how often each `yes` unit can be filled across the 306 WO-02 sample basins. Reads the local server (`/api/signature`, `/api/areas`), the local database and the saved WO-03 ring and flow tables (`output/edop/whg/q1_pairs.csv`, `q2_flow_pairs.csv`). Rules and templates are in `scripts/edop/whg/wo04_fill.py`. Read-only.

In [1]:
%matplotlib inline
# Cell 1 — Setup: server check, saved ring and flow tables, the places
import sys, time, warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.shared.db_utils import db_connect
from scripts.edop.whg import wo03_common as C
from scripts.edop.whg import wo04_fill as W

conn = db_connect()
PAIRS = pd.read_csv(C.OUT / 'q1_pairs.csv', dtype={'tag': str})
FLOW = pd.read_csv(C.OUT / 'q2_flow_pairs.csv', dtype={'tag': str})
SAMPLE = C.population_points()
health = W.get_json('/api/health')
print('commit', C.commit(), '| server:', health, '| pairs:', len(PAIRS), '| flow rows:', len(FLOW), '| test places:', len(C.PLACES), '| sample points:', len(SAMPLE))

commit 00b7484 | server: {'status': 'ok'} | pairs: 1739 | flow rows: 1739 | test places: 10 | sample points: 306


In [2]:
# Cell 2 — The ten test places: facts, units, and the filled_profiles (written to docs/design/dashboard/wo04_filled_profiles.md)
OUTMD = ROOT / 'docs' / 'design' / 'dashboard' / 'wo04_filled_profiles.md'
t0 = time.perf_counter()
TEST = {}
parts = ['# WO-04c: the dummy filled from real data, ten test places (filled_profiles)', '',
         'Generated by `notebooks/edop/whg/wo04c_fill.ipynb` (commit ' + C.commit() + '). Every `yes` unit of `modal_dummy_annotated.md` is filled from data; each `maybe` or `no` unit is a placeholder in the form `⟦id maybe: needs ...⟧`. Wording is the dummy\'s template, not polished. `S1.2a` names the basin\'s majority ecoregion; the template means the point\'s (see `wo04a_inventory.md`).', '',
         'Wording from `' + W.templates_stamp() + '`.', '']
for name, (lat, lon) in C.PLACES.items():
    F = W.build_facts(lat, lon, conn, PAIRS, FLOW)
    U = W.fill_units(F)
    TEST[name] = (F, U)
    parts.append(W.render_filled_profile(name, F, U))
    parts.append('---'); parts.append('')
OUTMD.write_text('\n'.join(parts))
print(f'wrote {OUTMD.name} | {W.templates_stamp()} | {time.perf_counter() - t0:.0f}s\n\n' + W.render_filled_profile('Timbuktu', *TEST['Timbuktu']))

wrote wo04_filled_profiles.md | profile_templates.md v1 (last commit none yet; edited since: yes) | 15s

## Timbuktu: environmental profile

### Setting
Timbuktu sits at about ⟦S1.1b maybe: needs live point elevation, or WHG's own⟧ m above sea level, in ⟦S1.1c maybe: needs terrain scheme or cutoff (wo04b)⟧ country. It lies in the Inner Niger Delta flooded savanna ecoregion, ⟦S1.2b maybe: needs stored one-line ecoregion gist⟧. ⟦S1.3 maybe: needs distance from the point to the next ecoregion⟧ ⟦S1.4 maybe: needs share cutoff for further ecoregions⟧

### Climate
Rainfall averages about 189 mm a year, ⟦S2.1b maybe: needs rule for "mostly in {months}"⟧. Temperatures average 28.6 °C, from 21.5 °C in January to 33.8 °C in June. ⟦S2.3 maybe: needs ring rank rules⟧

### Water
Timbuktu is ⟦S3.1 maybe: needs nearest-river layer: distance, name⟧. Most of the water passing here falls far upstream: the catchment above is 100 times the size of the local basin, and wetter than it. ⟦S3.3 maybe: needs cu

In [ ]:
# Cell 3 — Which yes units were filled at each test place (1 = filled, 0 = not)
UNITS = ['S1.2a', 'S2.1a', 'S2.2a', 'S2.2bc', 'S3.2a', 'S3.2b', 'S3.5', 'S4.2', 'S5.1a', 'S5.1b', 'S6.1a', 'S6.2', 'S7.2', 'S8.1a', 'S8.2', 'S9.2']
grid = pd.DataFrame({name: {u: int(U[u]['ok']) for u in UNITS} for name, (F, U) in TEST.items()})
print(grid.to_string())

In [ ]:
# Cell 4 — The sample: 306 basin interior points; saved to a CSV
rows, t_start = [], time.perf_counter()
for k, s in enumerate(SAMPLE, 1):
    F = W.build_facts(float(s['lat']), float(s['lon']), conn, PAIRS, FLOW)
    U = W.fill_units(F)
    d = dict(core_id=int(s['hybas_id_l6']), zone=s['climate_zone'], s32_class=U['S3.2b'].get('class', ''))
    for u in UNITS:
        d[u] = int(U[u]['ok']); d[u + '_na'] = int(U[u].get('na', False)); d[u + '_why'] = U[u]['why'] or ''
    rows.append(d)
    if k % 50 == 0:
        print(f'  {k} of {len(SAMPLE)} in {time.perf_counter() - t_start:.0f}s')
S4 = pd.DataFrame(rows)
S4.to_csv(C.OUT / 'wo04c_fill.csv', index=False)
print(f'rows saved: {len(S4)} | this run took {time.perf_counter() - t_start:.0f}s')

In [ ]:
# Cell 5 — Fill rate per yes unit over the sample: filled, not applicable (an optional part that does not fire), could not fill
L = [f'sample points: {len(S4)}']
for u in UNITS:
    n_ok = int(S4[u].sum()); n_na = int(S4[u + '_na'].sum())
    miss = S4[(S4[u] == 0) & (S4[u + '_na'] == 0)]
    why = miss[u + '_why'].value_counts().to_dict()
    L.append(f'{u}: filled {n_ok} | not applicable {n_na} | could not fill {len(miss)}' + (f' {why}' if why else ''))
L.append('')
L.append('S3.2b, the Analysis rule\'s class for each sample basin: ' + str(S4['s32_class'].value_counts().to_dict()))
print('\n'.join(L))